# 08 - The honest close: instability of learned reconstruction

Lecture section: 4.12  |  Spine note: this is the cautionary epilogue to the spine

$$\hat{x} = \arg\min_x\ \underbrace{D(Ax, y)}_{\text{data fidelity}} + \underbrace{R(x)}_{\text{prior}}\qquad\text{— what happens when we drop } D \text{ at test time?}$$

Learned reconstructors can reach high PSNR yet be **unstable**: a tiny, *worst-case* perturbation
of the measurements can cause a large, structured error -- the network erases real structure or
invents false structure (:footcite:`antun2020instabilities`). We demonstrate it honestly on the
most vulnerable architecture: a **direct** post-processing net $\hat{x} = G(A^\dagger y)$ that
*drops the physics* $D$ at inference (unlike PnP/unrolled, which keep data consistency). The key
comparison is **adversarial vs random** perturbations of the **same size**.

In [1]:
import torch
import tutorial_common as tc
import deepinv as dinv
from torch.utils.data import DataLoader

tc.set_seed()
N = 64

deepinv 0.4.1 | torch 2.9.1 | device cpu


## Train a direct post-processing network $G(A^\dagger y)$

It maps the filtered back-projection to a clean image. It is trained supervised on phantoms and
reaches a good clean-data PSNR -- but it never enforces $Ax \approx y$ at test time.

In [2]:
phys = tc.ct_physics(angles=40, sigma=0.02, size=N)
G = dinv.models.DnCNN(in_channels=1, out_channels=1, depth=7, pretrained=None).to(tc.DEVICE)

loader = DataLoader(dinv.utils.RandomPhantomDataset(size=N, length=32), batch_size=4, shuffle=True)
opt = torch.optim.Adam(G.parameters(), lr=1e-3)
G.train()
for epoch in range(40):
    for xb in loader:
        xb = xb.to(tc.DEVICE)
        rec = G(phys.fbp(phys(xb)))            # G(FBP(noisy y))
        loss = ((rec - xb) ** 2).mean()
        opt.zero_grad(); loss.backward(); opt.step()
G.eval()

x = tc.load_hero(N)
y = phys.A(x)                                   # clean measurement (isolate the perturbation effect)
with torch.no_grad():
    rec_clean = G(phys.fbp(y))
print(f"clean-data PSNR: {tc.psnr(rec_clean, x):.2f} dB")

clean-data PSNR: 19.94 dB


## Adversarial vs random perturbation of the same norm

We search for a small perturbation $\delta$ of the measurements that **maximises** the change in
the reconstruction, $\|G(A^\dagger(y+\delta)) - G(A^\dagger y)\|$, subject to a norm budget. Then
we compare against a **random** $\delta$ of the *identical* norm.

In [3]:
def adversarial_delta(frac, steps=60):
    """PGD ascent for a measurement perturbation of norm = frac * ||y||."""
    eps = frac * y.norm()
    delta = torch.zeros_like(y, requires_grad=True)
    for _ in range(steps):
        rec = G(phys.fbp(y + delta))
        loss = ((rec - rec_clean.detach()) ** 2).sum()      # maximise output change
        (grad,) = torch.autograd.grad(loss, delta)
        with torch.no_grad():
            delta += (eps / 8) * grad.sign()                # signed-gradient step...
            norm = delta.norm()
            if norm > eps:                                  # ...projected back into the norm ball
                delta *= eps / norm
        delta.requires_grad_(True)
    return delta.detach()

frac = 0.15                                                 # perturbation budget = 15% of ||y||
delta_adv = adversarial_delta(frac)
g = torch.Generator(device=tc.DEVICE).manual_seed(0)
delta_rnd = torch.randn(y.shape, generator=g, device=tc.DEVICE)
delta_rnd *= delta_adv.norm() / delta_rnd.norm()           # SAME norm as the adversarial one

with torch.no_grad():
    rec_adv = G(phys.fbp(y + delta_adv))
    rec_rnd = G(phys.fbp(y + delta_rnd))

chg_adv = ((rec_adv - rec_clean).norm() / rec_clean.norm()).item()
chg_rnd = ((rec_rnd - rec_clean).norm() / rec_clean.norm()).item()
print(f"||delta||/||y|| = {(delta_adv.norm()/y.norm()).item():.2f} (same for both)")
print(f"output change: adversarial {chg_adv:.3f}  vs  random {chg_rnd:.3f}  ({chg_adv/chg_rnd:.1f}x)")

tc.save_images(
    [x, rec_clean, rec_adv, rec_rnd],
    titles=[
        "x (ground truth)",
        tc.title_psnr("clean data", rec_clean, x),
        tc.title_psnr("adversarial perturbation", rec_adv, x),
        tc.title_psnr("random perturbation", rec_rnd, x),
    ],
    fname="08_instability.png",
    suptitle="Same-size perturbation: the worst-case one wrecks the reconstruction; random barely hurts",
)

||delta||/||y|| = 0.15 (same for both)
output change: adversarial 0.455  vs  random 0.183  (2.5x)


saved /Users/jonathan/Code/deepinv/lecture-tutorials/figures/08_instability.png


## Sensitivity curve: worst-case grows much faster than average-case

Across perturbation budgets, the adversarial change is consistently several times the random
change of equal norm -- the signature of an unstable reconstructor.

In [4]:
budgets = [0.025, 0.05, 0.10, 0.15, 0.20]
adv_curve, rnd_curve = [], []
for f in budgets:
    d = adversarial_delta(f)
    gen = torch.Generator(device=tc.DEVICE).manual_seed(1)
    r = torch.randn(y.shape, generator=gen, device=tc.DEVICE); r *= d.norm() / r.norm()
    with torch.no_grad():
        adv_curve.append(((G(phys.fbp(y + d)) - rec_clean).norm() / rec_clean.norm()).item())
        rnd_curve.append(((G(phys.fbp(y + r)) - rec_clean).norm() / rec_clean.norm()).item())
print("adversarial:", [round(v, 3) for v in adv_curve])
print("random:     ", [round(v, 3) for v in rnd_curve])

tc.save_curves(
    {"adversarial (worst-case)": (budgets, adv_curve), "random (same norm)": (budgets, rnd_curve)},
    fname="08_sensitivity.png", xlabel=r"perturbation budget  $\|\delta\|/\|y\|$",
    ylabel="relative output change", title="Instability: worst-case vs random perturbations",
    logy=False, markers=True,
)

adversarial: [0.072, 0.144, 0.293, 0.455, 0.612]
random:      [0.031, 0.061, 0.122, 0.19, 0.269]


saved /Users/jonathan/Code/deepinv/lecture-tutorials/figures/08_sensitivity.png


## Takeaway

High PSNR on clean data does **not** mean a reconstruction is trustworthy. A direct learned
inverse that ignores the physics is sensitive to small, worst-case measurement perturbations that
random noise of the same size does not cause. **Mitigations seen earlier in this lecture:** keep
the physics in the loop (data-consistency, as in PnP/unrolled), and report **uncertainty**
(notebook 7) so the model can say *where* -- and how much -- it is unsure.